# **Data Cleaning and Processing for Producer's Direct and Global Historical Climatology Network daily Data Sets**

In [ ]:
import gc
import os
import gcsfs
import duckdb
import tarfile
import pyarrow as pa
import pandas as pd
import google.auth

from tqdm import tqdm
from io import BytesIO
from pathlib import Path
from google.colab import drive
from google.oauth2 import service_account

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
key_path = '/content/drive/MyDrive/final-project-210518-c4be6f6bc63b.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path

fs = gcsfs.GCSFileSystem()

**Producer's Direct Data Set CSV file**

In [ ]:
# Converting Producer's Direct data to a parquet file.

data_file_path = '/content/drive/MyDrive/b0cd514b-b9cc-4972-a0c2-c91726e6d825.csv'
parquet_file_path = '/content/drive/MyDrive/b0cd514b-b9cc-4972-a0c2-c91726e6d825.parquet'

conn = duckdb.connect()
conn.execute("SET threads = 1;")
conn.execute("SET memory_limit = '8GB';")

conn.sql(f"COPY (SELECT * FROM '{data_file_path}') TO '{parquet_file_path}' (FORMAT 'PARQUET');")
conn.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Global Historical Climatology Network daily (GHCNd) Stations TXT and  GZIP-compressed TAR file**

In [ ]:
# Cleaning the GHCND stations txt file.

# Information about the format of the txt file was found on the GHCND README txt: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt

ghcnd_stations_txt_path = '/content/drive/MyDrive/ghcnd-stations.txt'

col_specs = [(0, 11),
            (12, 20),
            (21, 30),
            (31, 37),
            (38, 40),
            (41, 71),
            (72, 75),
            (76, 79),
            (80, 85)]

stations_txt_df = pd.read_fwf(ghcnd_stations_txt_path, colspecs=col_specs, names=['ID','Latitude', 'Longitude', 'Altitude', 'State', 'Station Details', 'GSN Flag', 'HCN/CRN Flag', 'WMO ID'])
stations_txt_df

,ID,Latitude,Longitude,Altitude,State,Station Details,GSN Flag,HCN/CRN Flag,WMO ID
0,ACW00011604,17.1167,-61.7833,10.1,NaN,ST JOHNS COOLIDGE FLD,NaN,NaN,NaN
1,ACW00011647,17.1333,-61.7833,19.2,NaN,ST JOHNS,NaN,NaN,NaN
2,AE000041196,25.3330,55.5170,34.0,NaN,SHARJAH INTER. AIRP,GSN,NaN,41196.0
3,AEM00041194,25.2550,55.3640,10.4,NaN,DUBAI INTL,NaN,NaN,41194.0
4,AEM00041217,24.4330,54.6510,26.8,NaN,ABU DHABI INTL,NaN,NaN,41217.0
...,...,...,...,...,...,...,...,...,...
129653,ZI000067969,-21.0500,29.3670,861.0,NaN,WEST NICHOLSON,NaN,NaN,67969.0
129654,ZI000067975,-20.0670,30.8670,1095.0,NaN,MASVINGO,NaN,NaN,67975.0
129655,ZI000067977,-21.0170,31.5830,430.0,NaN,BUFFALO RANGE,NaN,NaN,67977.0
129656,ZI000067983,-20.2000,32.6160,1132.0,NaN,CHIPINGE,GSN,NaN,67983.0


In [ ]:
# Filtering for Kenya, Tanzania, and Uganda's stations by ensuring both the ID and Geolocations are within the range of these countries.

kenyalatmin, kenyalatmax, kenyalonmin, kenyalonmax = [-4.5, 4.5, 34.0, 42.0]
tanzlatmin, tanzlatmax, tanzlonmin, tanzlonmax = [-12.0, -1.0, 21.0, 41.0]
uglatmin, uglatmax, uglonmin, uglonmax = [0.35, 4.5, 29.5, 35.0]

kenya_stat_id = stations_txt_df[stations_txt_df['ID'].str.startswith('KE')]
tanz_stat_id = stations_txt_df[stations_txt_df['ID'].str.startswith('TZ')]
ug_stat_id = stations_txt_df[stations_txt_df['ID'].str.startswith('UG')]


kenya_stations = set(kenya_stat_id[(kenya_stat_id['Latitude'] >= kenyalatmin) & (kenya_stat_id['Latitude'] <= kenyalatmax) & (kenya_stat_id['Longitude'] >= kenyalonmin) & (kenya_stat_id['Longitude'] <= kenyalonmax)]['ID'].tolist())
tanzania_stations = set(tanz_stat_id[(tanz_stat_id['Latitude'] >= tanzlatmin) & (tanz_stat_id['Latitude'] <= tanzlatmax) & (tanz_stat_id['Longitude'] >= tanzlonmin) & (tanz_stat_id['Longitude'] <= tanzlonmax)]['ID'].tolist())
ug_stations = set(ug_stat_id[(ug_stat_id['Latitude'] >= uglatmin) & (ug_stat_id['Latitude'] <= uglatmax) & (ug_stat_id['Longitude'] >= uglonmin) & (ug_stat_id['Longitude'] <= uglonmax)]['ID'].tolist())

all_stations_id = kenya_stations.union(tanzania_stations).union(ug_stations)
all_stations_id

{'KE000063612',
 'KE000063619',
 'KE000063624',
 'KE000063661',
 'KE000063723',
 'KE000063740',
 'KE000063820',
 'KEM00063686',
 'KEM00063741',
 'KEM00063799',
 'TZ000063729',
 'TZ000063733',
 'TZ000063756',
 'TZ000063789',
 'TZ000063790',
 'TZ000063791',
 'TZ000063816',
 'TZ000063832',
 'TZ000063862',
 'TZ000063887',
 'TZ000063894',
 'TZ000063962',
 'TZ000063971',
 'TZM00063870',
 'UG000063602',
 'UG000063630',
 'UG000063654',
 'UG000063682',
 'UG000063684'}

In [ ]:
# Creating a parquet file of the filtered stations in all_stations_id, as they contain information for Kenya, Tanzania, Uganda.
# While creating a parquet file, method parses through the ghcnd dly files and formats each file correctly.

# Information about dly files formatting was found on the GHCND README txt: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt

gcs_tar_path = 'gs://datakind-ghcnd-weather-data/ghcnd_all.tar.gz'
gcs_output_parquet_path = 'gs://datakind-ghcnd-weather-data/ghcnd_all_parsed.parquet'

def parse_dly_files(dly_file):
    dly_file_lines = dly_file.strip().split('\n')
    parsed_data = []

    for line in dly_file_lines:
      # using a fixed-width file approach from looking at the GHCND README txt
        station = line[0:11]
        year = int(line[11:15])
        month = int(line[15:17])
        element = line[17:21]

        # starting character index of the first day
        day_data_start = 21

        for day in range(1, 32):
          data_block = line[day_data_start:day_data_start + 8]

          if len(data_block.strip()) > 0:
                value_str = data_block[0:5].strip()
                m_flag = data_block[5:6].strip()
                q_flag = data_block[6:7].strip()
                s_flag = data_block[7:8].strip()

          if value_str and value_str != '-9999':
                    try:
                        value = int(value_str)
                        date_str = f"{year:04d}{month:02d}{day:02d}"
                        parsed_data.append([station, date_str, element, value, m_flag, q_flag, s_flag])

                    except ValueError:
                        continue

          day_data_start += 8

    return parsed_data

def convert_dly_to_parquet_filtered(gcs_tar_path, gcs_output_parquet_path, filtered_station_ids):
    all_parsed_data = []

    print("Starting filtered data parsing...")
    # fs.connects to gcs, opens the file in the gcs bucket and returns a file-like object
    with fs.open(gcs_tar_path, 'rb') as gcs_file:
      # tarfile.open recieves the gcs file and opens it, ensuring to decompress the gzip file while reading
        with tarfile.open(fileobj=gcs_file, mode='r:gz') as tar:
          # iterating through each 'member'/file in the tar file and using tqdm to display a progress bar for monitoring
           for member in tqdm(tar, desc="Parsing filtered stations"):
                # need to ensure file is a dly file
                if member.isfile() and member.name.endswith('.dly'):
                    filename = os.path.basename(member.name)
                    member_station_id = filename.replace('.dly', '')
                    # need to ensure dly file is from the stations we want (from the filtered list in all_stations id)
                    if member_station_id in filtered_station_ids:
                        f = tar.extractfile(member)
                        if f:
                            file_content = f.read().decode('utf-8')
                            all_parsed_data.extend(parse_dly_files(file_content))

    print(f"\nParsing complete. Total records: {len(all_parsed_data)}")

    if not all_parsed_data:
        print("No data was parsed.")
        return
    # using PyArrow to establish appropriate schema for the data that will be written to a Parquet file.
    schema = pa.schema([
        pa.field('STATION', pa.string()), pa.field('DATE', pa.string()),
        pa.field('ELEMENT', pa.string()), pa.field('VALUE', pa.int32()),
        pa.field('M_FLAG', pa.string()), pa.field('Q_FLAG', pa.string()),
        pa.field('S_FLAG', pa.string())
    ])

    arrow_table = pa.Table.from_arrays([
        pa.array(col) for col in zip(*all_parsed_data)
    ], schema=schema)

    # Deleting RAM memory
    del all_parsed_data
    gc.collect()

    print("Connecting to DuckDB and writing to Parquet...")
    conn = duckdb.connect(database=':memory:', read_only=False)
    conn.sql("INSTALL httpfs;")
    conn.sql("LOAD httpfs;")

    hmac_access_id = #hmac access id removed
    hmac_secret = #hmac secret id removed

    conn.sql(f"CREATE OR REPLACE SECRET secret (TYPE gcs, KEY_ID '{hmac_access_id}', SECRET '{hmac_secret}');")
    conn.sql("SET enable_object_cache = true;")

    conn.register('arrow_table', arrow_table)
    conn.sql(f"COPY arrow_table TO '{gcs_output_parquet_path}' (FORMAT 'parquet', COMPRESSION 'snappy', OVERWRITE 'true')")
    conn.close()
    print(f"Successfully converted and saved filtered .dly data to {gcs_output_parquet_path}")

convert_dly_to_parquet_filtered(gcs_tar_path, gcs_output_parquet_path, all_stations_id)

Starting filtered data parsing...


Parsing filtered stations: 129597it [12:27, 173.34it/s]



Parsing complete. Total records: 924491
Connecting to DuckDB and writing to Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully converted and saved filtered .dly data to gs://datakind-ghcnd-weather-data/ghcnd_all_parsed.parquet
